# Difference-in-Differences: Applications

**P@S 2026, Lecture 12 (Mar 2)**

On Wednesday we built the DiD toolkit (2x2 table, TWFE, event study). Today we apply it to real papers about media and politics:

| Paper | Treatment | Outcome |
|:------|:----------|:--------|
| **Kim & Kam (2023)** | COVID-19 pandemic | Anti-Chinese Yelp reviews |
| **DellaVigna & Kaplan (2007)** | Fox News cable access | Republican vote share |

**What we'll do:**
1. **Parts 1-8**: Simulate a dataset inspired by Kim & Kam (2023) and run the full DiD toolkit on it
2. **Part 9**: Summary comparing Wednesday's concepts to today's applications
3. **Part 10**: The big picture: RCT, DiD, and RDD all use OLS
4. **Part 11**: Replicate DellaVigna & Kaplan (2007) using the **actual data** from the paper

By the end, you'll have applied DiD to both simulated and real data, and seen what replication looks like in practice.

## Reproducibility: what's real, what's simulated, and what we learned

This notebook uses two kinds of data. It's important to be explicit about which is which and why.

### What's simulated (Parts 1-10)

**Kim & Kam (2023)** studied anti-Chinese bias in Yelp reviews during COVID-19. Their original data (scraped Yelp reviews from 8 metro areas) is not publicly available. So we **simulate** a dataset that mimics the structure of their study:

- 200 restaurants (100 Chinese, 100 American) across 8 metros, observed monthly
- A treatment shock at month 15 (March 2020 lockdown)
- A -0.15 star treatment effect on Chinese restaurants (close to their actual estimate of -0.13)
- The data-generating process is intentionally simple so you can see DiD recover a *known* effect

### What's real (Part 11)

**DellaVigna & Kaplan (2007)** posted their replication data on Berkeley's website. Part 11 downloads the actual dataset (10,126 towns, 211 variables) and replicates their main results. This is real data from a real paper.

### Five lessons from the replication

1. **Simpson's paradox is real.** The raw difference shows Fox News towns shifting *less* Republican (-1.2pp). With census controls, the sign flips to +0.4 to +0.7pp. Urban towns got Fox News first, and urban towns were already trending Democratic. Without controls, the confounder dominates.

2. **Weighted least squares matters.** DK weight by total votes cast in 1996. This makes larger towns count more, which changes point estimates noticeably. If you run OLS without weights, you get different numbers.

3. **Clustered standard errors matter.** DK cluster at the level of the cable system (`account2000`), which is the unit of treatment assignment. Clustering roughly doubles the standard errors compared to OLS defaults. The effect is still significant, but the confidence intervals are much wider.

4. **Missing data in placebo outcomes.** The placebo test variables have names like `reppresfv2p96m92` and `reppresfv2p92m88`. These are Stata-style variable names from a `.dta` file posted for replication in 2007. The naming convention: `rep` (Republican) + `pres` (presidential) + `fv2p` (fraction of two-party vote) + `96m92` (1996 minus 1992). Stata has historically had strict variable name limits (8 characters in early versions), so economists developed a culture of cramming everything into one lowercase slug. These names are frozen in time because renaming them would break the code that matches the paper. The variables have missing values for some towns (not all towns have vote data going back to 1988). Stata drops these silently; Python's statsmodels needs explicit `dropna()` to avoid a dimension mismatch in the clustered SE computation. A small thing, but the kind of thing that breaks replications.

5. **Our estimates match.** With census controls, we get a Fox News effect of approximately +0.004 to +0.007 percentage points, consistent with DK's Table IV columns (1)-(3). The placebo tests (1992-1996, 1988-1992) show no significant Fox News effect, as expected.

## Part 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
np.random.seed(42)

## Part 2: Simulate the Yelp ratings data

We simulate a dataset inspired by Kim and Kam (2023). The key elements:
- **Two types of restaurants**: Chinese and American
- **Time period**: 24 months (Jan 2019 - Dec 2020)
- **Treatment shock**: COVID-19 lockdown (March 2020, month 15)
- **Treatment effect**: Chinese restaurants receive lower ratings after COVID, American restaurants don't

We simulate 200 restaurants (100 Chinese, 100 American) across 8 metro areas, each receiving monthly review ratings.

In [ ]:
# Simulate restaurant-month panel data
n_restaurants = 200
n_months = 24
lockdown_month = 15  # March 2020

# Restaurant attributes
restaurant_ids = np.arange(n_restaurants)
cuisine = np.array(['Chinese'] * 100 + ['American'] * 100)
metro = np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix', 'Philly', 'Dallas', 'SF'],
                         size=n_restaurants)

# Restaurant-level baseline rating (random intercept)
base_rating = np.random.normal(3.9, 0.3, n_restaurants)

# Build panel
rows = []
for i in range(n_restaurants):
    for m in range(1, n_months + 1):
        # Common time trend: slight upward drift for all restaurants
        time_trend = 0.005 * m

        # Treatment effect: Chinese restaurants get lower ratings after lockdown
        treat_effect = 0.0
        if cuisine[i] == 'Chinese' and m >= lockdown_month:
            treat_effect = -0.15  # About 0.15 stars lower

        # General COVID bump (American restaurants got slightly better reviews post-lockdown)
        covid_bump = 0.08 if m >= lockdown_month else 0.0

        # Rating with noise
        rating = base_rating[i] + time_trend + treat_effect + covid_bump + np.random.normal(0, 0.25)
        rating = np.clip(rating, 1, 5)

        rows.append({
            'restaurant_id': i,
            'cuisine': cuisine[i],
            'metro': metro[i],
            'month': m,
            'post_lockdown': int(m >= lockdown_month),
            'chinese': int(cuisine[i] == 'Chinese'),
            'rating': round(rating, 2)
        })

df = pd.DataFrame(rows)
print(f"Dataset: {len(df)} restaurant-month observations")
print(f"Restaurants: {df['restaurant_id'].nunique()} ({df[df['chinese']==1]['restaurant_id'].nunique()} Chinese, {df[df['chinese']==0]['restaurant_id'].nunique()} American)")
print(f"Months: {df['month'].nunique()} (lockdown at month {lockdown_month})")
df.head()

## Part 3: Visualize the data

Plot average ratings over time for Chinese and American restaurants. This is the analogue of Kim and Kam's Figure 1.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Average ratings by cuisine and month
for cuisine_type, color, marker in [('Chinese', '#c0392b', 'o'), ('American', '#2980b9', 's')]:
    group = df[df['cuisine'] == cuisine_type].groupby('month')['rating'].mean()
    ax.plot(group.index, group.values, f'{marker}-', color=color, linewidth=2,
            markersize=6, label=f'{cuisine_type} restaurants')

# Lockdown line
ax.axvline(x=lockdown_month, color='gray', linestyle='--', linewidth=2, alpha=0.7)
ax.text(lockdown_month + 0.3, ax.get_ylim()[0] + 0.01, 'COVID-19\nlockdown',
        fontsize=11, color='gray', va='bottom')

# Month labels
month_labels = ['Jan 19', '', 'Mar', '', 'May', '', 'Jul', '', 'Sep', '', 'Nov', '',
                'Jan 20', '', 'Mar', '', 'May', '', 'Jul', '', 'Sep', '', 'Nov', 'Dec']
ax.set_xticks(range(1, 25))
ax.set_xticklabels(month_labels, fontsize=8, rotation=45)
ax.set_ylabel('Average Yelp Rating', fontsize=12)
ax.set_title('Yelp Ratings: Chinese vs. American Restaurants (Simulated)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Before lockdown, ratings for both cuisines track together (parallel trends).")
print("After lockdown, Chinese restaurant ratings drop while American ratings stay stable or rise.")

## Part 4: DiD by hand (the 2x2 table)

Same arithmetic from Wednesday, new context.

In [ ]:
# 2x2 table
cn_pre  = df[(df['chinese'] == 1) & (df['post_lockdown'] == 0)]['rating'].mean()
cn_post = df[(df['chinese'] == 1) & (df['post_lockdown'] == 1)]['rating'].mean()
am_pre  = df[(df['chinese'] == 0) & (df['post_lockdown'] == 0)]['rating'].mean()
am_post = df[(df['chinese'] == 0) & (df['post_lockdown'] == 1)]['rating'].mean()

print("              Pre-lockdown   Post-lockdown   Change")
print(f"Chinese:      {cn_pre:.3f}         {cn_post:.3f}          {cn_post - cn_pre:+.3f}")
print(f"American:     {am_pre:.3f}         {am_post:.3f}          {am_post - am_pre:+.3f}")
print(f"\nDiD estimate: ({cn_post - cn_pre:.3f}) - ({am_post - am_pre:.3f}) = {(cn_post - cn_pre) - (am_post - am_pre):.3f}")
print(f"\nChinese restaurants received ratings about {abs((cn_post - cn_pre) - (am_post - am_pre)):.2f} stars lower")
print("after COVID, relative to American restaurants.")

## Part 5: TWFE regression

The regression version: restaurant fixed effects + month fixed effects + treatment indicator.

In [ ]:
# Create treatment variable: Chinese x PostLockdown
df['treated'] = df['chinese'] * df['post_lockdown']

# TWFE with restaurant and month fixed effects, clustered by restaurant
model = smf.ols('rating ~ treated + C(restaurant_id) + C(month)', data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['restaurant_id']}
)

print("TWFE Regression: rating ~ treated + restaurant FE + month FE")
print(f"  DiD estimate (treated):  {model.params['treated']:.4f}")
print(f"  Clustered SE:            {model.bse['treated']:.4f}")
print(f"  t-statistic:             {model.tvalues['treated']:.2f}")
print(f"  p-value:                 {model.pvalues['treated']:.6f}")
print(f"\nThe regression recovers the treatment effect: Chinese restaurants")
print(f"received ratings about {abs(model.params['treated']):.2f} stars lower after the lockdown.")

## Part 6: The interaction form

Writing it out explicitly, as in the slides:

$$Y_{it} = \alpha + \beta_1 \cdot \text{Chinese} + \beta_2 \cdot \text{PostLockdown} + \beta_3 \cdot (\text{Chinese} \times \text{PostLockdown}) + \varepsilon_{it}$$

$\beta_3$ is the DiD estimate.

In [ ]:
# Interaction form
model_int = smf.ols('rating ~ chinese * post_lockdown', data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['restaurant_id']}
)

print("Interaction form: rating ~ Chinese + PostLockdown + Chinese:PostLockdown\n")
print(model_int.summary().tables[1])
print(f"\nMapping to the 2x2 table:")
print(f"  Intercept (American, pre):      {model_int.params['Intercept']:.3f}")
print(f"  Chinese (level difference):     {model_int.params['chinese']:+.3f}")
print(f"  PostLockdown (common trend):    {model_int.params['post_lockdown']:+.3f}")
print(f"  Chinese:PostLockdown (DiD):     {model_int.params['chinese:post_lockdown']:+.3f}")

## Part 7: Specificity test

Kim and Kam's key insight: if this is really anti-Chinese bias (not general pandemic disruption), then other Asian cuisines should NOT be affected. Let's add Japanese restaurants to our simulation and test.

In [ ]:
# Add Japanese, Korean, and Thai restaurants (no treatment effect)
extra_cuisines = ['Japanese', 'Korean', 'Thai']
extra_rows = []
for cuisine_name in extra_cuisines:
    for i in range(50):  # 50 restaurants per cuisine
        rid = n_restaurants + i + extra_cuisines.index(cuisine_name) * 50
        base = np.random.normal(3.9, 0.3)
        for m in range(1, n_months + 1):
            time_trend = 0.005 * m
            covid_bump = 0.08 if m >= lockdown_month else 0.0
            # NO anti-cuisine treatment effect for these cuisines
            rating = base + time_trend + covid_bump + np.random.normal(0, 0.25)
            rating = np.clip(rating, 1, 5)
            extra_rows.append({
                'restaurant_id': rid,
                'cuisine': cuisine_name,
                'metro': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix', 'Philly', 'Dallas', 'SF']),
                'month': m,
                'post_lockdown': int(m >= lockdown_month),
                'chinese': 0,
                'rating': round(rating, 2),
                'treated': 0
            })

df_all = pd.concat([df, pd.DataFrame(extra_rows)], ignore_index=True)

# Run DiD for each cuisine vs American
print("Specificity test: DiD estimate for each Asian cuisine vs. American restaurants\n")
print(f"{'Cuisine':<12} {'DiD estimate':>12}  {'SE':>8}  {'p-value':>8}")
print("-" * 45)

for cuisine_name in ['Chinese', 'Japanese', 'Korean', 'Thai']:
    sub = df_all[df_all['cuisine'].isin([cuisine_name, 'American'])].copy()
    sub['is_asian'] = (sub['cuisine'] == cuisine_name).astype(int)
    sub['treat'] = sub['is_asian'] * sub['post_lockdown']
    m = smf.ols('rating ~ is_asian * post_lockdown', data=sub).fit(
        cov_type='cluster', cov_kwds={'groups': sub['restaurant_id']}
    )
    coef = m.params['is_asian:post_lockdown']
    se = m.bse['is_asian:post_lockdown']
    pval = m.pvalues['is_asian:post_lockdown']
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f"{cuisine_name:<12} {coef:>+12.4f}  {se:>8.4f}  {pval:>8.4f} {sig}")

print("\nOnly Chinese restaurants show a significant negative effect.")
print("This specificity supports the 'othering' interpretation.")

## Part 8: Event study

Let the treatment effect vary by month. Pre-treatment coefficients should be near zero (parallel trends test); post-treatment coefficients show the effect dynamics.

In [ ]:
# Event study: interact Chinese with each month (omit month 14 = Feb 2020 as reference)
df['rel_month'] = df['month'] - lockdown_month  # 0 = lockdown month

# Create period dummies (quarterly bins for cleaner plot)
df['quarter'] = pd.cut(df['rel_month'],
                       bins=[-15, -9, -6, -3, -1, 0, 3, 6, 10],
                       labels=['Q-4', 'Q-3', 'Q-2', 'Q-1(ref)', 'Q0', 'Q+1', 'Q+2', 'Q+3'])

# Use month-level interactions for a proper event study
ref_month = 14  # Feb 2020, one month before lockdown
event_months = [m for m in range(1, 25) if m != ref_month]

formula_parts = []
for m in event_months:
    col = f'cn_m{m}'
    df[col] = ((df['month'] == m) & (df['chinese'] == 1)).astype(int)
    formula_parts.append(col)

formula = 'rating ~ ' + ' + '.join(formula_parts) + ' + C(restaurant_id) + C(month)'

# Fit (this may take a moment with 200 restaurant fixed effects)
event_model = smf.ols(formula, data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['restaurant_id']}
)

# Extract coefficients
months_plot = list(range(1, 25))
coefs = []
ses = []
for m in months_plot:
    col = f'cn_m{m}'
    if m == ref_month:
        coefs.append(0)
        ses.append(0)
    else:
        coefs.append(event_model.params[col])
        ses.append(event_model.bse[col])

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
rel_months = [m - lockdown_month for m in months_plot]

ax.errorbar(rel_months, coefs, yerr=[1.96 * s for s in ses],
            fmt='o', color='#2c3e50', markersize=5, capsize=3, capthick=1.5, linewidth=1)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8)
ax.axvline(x=0, color='#c0392b', linestyle='--', linewidth=2, alpha=0.7)

ax.fill_between([-15, -0.5], -0.5, 0.5, alpha=0.03, color='blue')
ax.fill_between([-0.5, 10], -0.5, 0.5, alpha=0.03, color='red')
ax.text(-7, 0.35, 'Pre-lockdown\n(should be near 0)', fontsize=11, color='#2980b9', ha='center')
ax.text(5, 0.35, 'Post-lockdown\n(treatment effect)', fontsize=11, color='#c0392b', ha='center')

ax.set_xlabel('Months relative to COVID lockdown (March 2020 = 0)', fontsize=12)
ax.set_ylabel('Estimated coefficient (Chinese x month)', fontsize=12)
ax.set_title('Event Study: Anti-Chinese Bias in Yelp Ratings', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Pre-lockdown coefficients cluster around zero (parallel trends holds).")
print("Post-lockdown coefficients are negative (Chinese restaurants penalized).")

## Part 9: Summary

| Step | Wednesday (Concepts) | Today (Applications) |
|------|---------------------|---------------------|
| **Data** | Organ donation (real) | Yelp ratings (simulated, inspired by Kim & Kam) |
| **Treated** | California (active choice) | Chinese restaurants |
| **Control** | 25 other states | American restaurants |
| **DiD estimate** | -0.022 (reduced donation) | ~-0.15 stars (lower ratings) |
| **Specificity** | N/A | Only Chinese, not Japanese/Korean/Thai |
| **Event study** | Flat pre, negative post | Flat pre, negative post |

**The same DiD toolkit works across domains.** The method doesn't care whether the outcome is organ donation rates or Yelp stars. What matters is: (1) a clear treatment shock, (2) a plausible control group, (3) parallel trends before treatment.

## Part 10: The big picture — it's all OLS

Every causal inference method we've seen in this course uses OLS (ordinary least squares) as the computational engine. What changes is the **identification strategy**: the argument for why $\hat{\beta}$ estimates a causal effect.

| Method | Identification strategy | The regression | Course example |
|:-------|:----------------------|:---------------|:---------------|
| **Randomized experiment** | Random assignment breaks confounding | $Y = \alpha + \beta T + \varepsilon$ | Allcott (2020): Facebook deactivation |
| **Difference-in-differences** | Parallel trends + treatment timing | $Y = \alpha + \gamma_{\text{group}} + \delta_{\text{time}} + \beta(\text{Group} \times \text{Post}) + \varepsilon$ | Kim & Kam (2023): Yelp ratings |
| **Regression discontinuity** | Continuity at a sharp cutoff | $Y = \alpha + \beta T + f(X) + \varepsilon$ | *(next week)* |

**The punchline:** $\beta$ is always estimated the same way — minimize $\sum (Y_i - \hat{Y}_i)^2$. The "method" is the reason you believe $\beta$ is causal, not just correlational.

In [ ]:
# Part 10 demo: all three methods use sm.OLS under the hood

# --- Method 1: Randomized experiment (simple treatment indicator) ---
# Y = alpha + beta * T + epsilon
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(123)
n = 500
T_exp = np.random.binomial(1, 0.5, n)
Y_exp = 2.0 + 0.5 * T_exp + np.random.normal(0, 1, n)
X_exp = sm.add_constant(T_exp)
m1 = sm.OLS(Y_exp, X_exp).fit()

# --- Method 2: Difference-in-differences (from our simulation above) ---
# Y = alpha + gamma_group + delta_time + beta * (Group x Post) + epsilon
m2 = smf.ols('rating ~ chinese * post_lockdown', data=df).fit()

# --- Method 3: Regression discontinuity (simulated) ---
# Y = alpha + beta * T + f(X) + epsilon
np.random.seed(456)
X_rd = np.random.uniform(-1, 1, 1000)
T_rd = (X_rd >= 0).astype(int)
Y_rd = 1.0 + 0.8 * T_rd + 2.0 * X_rd + np.random.normal(0, 0.5, 1000)
X_mat = np.column_stack([np.ones(1000), T_rd, X_rd])
m3 = sm.OLS(Y_rd, X_mat).fit()

# Print the key coefficient from each
print("Method                    beta_hat    SE        Engine")
print("-" * 60)
print(f"Randomized experiment     {m1.params[1]:+.4f}    {m1.bse[1]:.4f}    sm.OLS")
print(f"Diff-in-differences       {m2.params['chinese:post_lockdown']:+.4f}    {m2.bse['chinese:post_lockdown']:.4f}    smf.ols (= sm.OLS)")
print(f"Regression discontinuity  {m3.params[1]:+.4f}    {m3.bse[1]:.4f}    sm.OLS")
print()
print("Same function. Different identification arguments.")

## Part 11: Replicating DellaVigna & Kaplan (2007) with real data

The previous parts used **simulated** data inspired by Kim & Kam (2023). Now we replicate a classic DiD study using the **actual data** from:

> DellaVigna, S. and E. Kaplan (2007). "The Fox News Effect: Media Bias and Voting." *Quarterly Journal of Economics* 122(3): 1187-1234.

**The question:** Did the introduction of Fox News on cable TV increase Republican vote share?

**The natural experiment:** Fox News launched in October 1996. By 2000, about 35% of US towns had cable access to Fox News. The key insight: cable operators decided whether to carry Fox News based on channel capacity and contracts, not local politics. This creates quasi-random variation in exposure.

**The DiD:** Compare the change in Republican presidential vote share (1996 to 2000) between towns that got Fox News and towns that didn't.

In [ ]:
# Download DellaVigna & Kaplan replication data from Berkeley
import os, zipfile, urllib.request
import pandas as pd
import numpy as np

data_dir = '/tmp/foxnews_data'
zip_path = os.path.join(data_dir, 'foxnews.zip')
dta_path = os.path.join(data_dir, 'FoxNewsFinalDataQJE.dta')

if not os.path.exists(dta_path):
    os.makedirs(data_dir, exist_ok=True)
    url = 'https://eml.berkeley.edu/~sdellavi/data/FoxNewsDataQJEMay07.zip'
    print(f'Downloading from {url}...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(data_dir)
    print('Done.')
else:
    print('Data already downloaded.')

dk = pd.read_stata(dta_path)
print(f'\nDataset: {dk.shape[0]:,} towns, {dk.shape[1]} variables')
print(f'Main sample (sample12000==1): {dk["sample12000"].sum():,.0f} towns')
print(f'Fox News towns: {dk.loc[dk["sample12000"]==1, "foxnews2000"].sum():,.0f} ({dk.loc[dk["sample12000"]==1, "foxnews2000"].mean()*100:.1f}%)')

In [ ]:
# Key variables
sample = dk[dk['sample12000'] == 1].copy()

print('=== Outcome variable ===')
print('reppresfv2p00m96: Change in Republican two-party vote share, 2000 minus 1996')
print(f'  Mean: {sample["reppresfv2p00m96"].mean():.4f}')
print(f'  SD:   {sample["reppresfv2p00m96"].std():.4f}')
print(f'  N:    {sample["reppresfv2p00m96"].notna().sum():,}')

print('\n=== Treatment variable ===')
print('foxnews2000: Whether town had Fox News on cable by 2000')
print(f'  Treated:   {(sample["foxnews2000"]==1).sum():,} towns')
print(f'  Untreated: {(sample["foxnews2000"]==0).sum():,} towns')

print('\n=== Simple comparison ===')
fox_yes = sample[sample['foxnews2000'] == 1]['reppresfv2p00m96'].mean()
fox_no  = sample[sample['foxnews2000'] == 0]['reppresfv2p00m96'].mean()
print(f'  Fox News towns:    {fox_yes:+.4f} (avg change in R vote share)')
print(f'  No Fox News towns: {fox_no:+.4f}')
print(f'  Raw difference:    {fox_yes - fox_no:+.4f}')
print('\n  Note: The raw difference is NEGATIVE (Fox News towns shifted LESS Republican).')
print('  This is Simpson\'s paradox: Fox News was more available in urban/Democratic areas.')
print('  We need controls to isolate the causal effect.')

### Replicating Table IV: The Fox News effect on Republican vote share

DellaVigna & Kaplan's main result is in Table IV. They regress the **change** in Republican vote share (2000 minus 1996) on Fox News availability, with increasingly rich controls:

1. **No controls** (weighted by 1996 turnout)
2. **Census controls** (population, education, race, income, urbanization, etc.)

The identification assumption: conditional on controls, Fox News availability is uncorrelated with unobserved determinants of vote share changes.

In [ ]:
import statsmodels.formula.api as smf

sample = dk[dk['sample12000'] == 1].copy()

# Census control variables (from the Stata .do file)
census_controls = [
    'pop2000', 'hs2000', 'hsp2000', 'college2000', 'male2000',
    'black2000', 'hisp2000', 'empl2000', 'unempl2000', 'married2000',
    'income2000', 'urban2000',
    # Changes 2000-1990
    'pop00m90', 'hs00m90', 'hsp00m90', 'college00m90', 'male00m90',
    'black00m90', 'hisp00m90', 'empl00m90', 'unempl00m90', 'married00m90',
    'income00m90', 'urban00m90'
]

# Verify all controls exist
available = [c for c in census_controls if c in sample.columns]

# --- Column 1: No controls, weighted ---
m1 = smf.wls('reppresfv2p00m96 ~ foxnews2000', data=sample,
             weights=sample['totpresvotes1996']).fit(
    cov_type='cluster', cov_kwds={'groups': sample['account2000']}
)

# --- Column 2: Census controls, weighted ---
formula2 = 'reppresfv2p00m96 ~ foxnews2000 + ' + ' + '.join(available)
m2 = smf.wls(formula2, data=sample,
             weights=sample['totpresvotes1996']).fit(
    cov_type='cluster', cov_kwds={'groups': sample['account2000']}
)

print('Table IV: Effect of Fox News on Republican Vote Share (2000-1996)')
print('=' * 65)
print(f'{"":<30} {"Col 1":>15} {"Col 2":>15}')
print(f'{"":30} {"(no controls)":>15} {"(census ctrl)":>15}')
print('-' * 65)
print(f'{"Fox News coefficient":<30} {m1.params["foxnews2000"]:>+15.4f} {m2.params["foxnews2000"]:>+15.4f}')
print(f'{"Clustered SE":<30} {m1.bse["foxnews2000"]:>15.4f} {m2.bse["foxnews2000"]:>15.4f}')
print(f'{"p-value":<30} {m1.pvalues["foxnews2000"]:>15.4f} {m2.pvalues["foxnews2000"]:>15.4f}')
print(f'{"N":<30} {int(m1.nobs):>15,} {int(m2.nobs):>15,}')
print(f'{"R-squared":<30} {m1.rsquared:>15.4f} {m2.rsquared:>15.4f}')
print('-' * 65)
print(f'\nWith census controls, Fox News increased R vote share by ~{m2.params["foxnews2000"]:.3f} pp.')
print('The paper reports 0.004-0.007 depending on specification.')
print('Without controls, the coefficient is negative (Simpson\'s paradox).')

### Placebo tests: Did Fox News "cause" vote changes before it existed?

A key validity check: Fox News launched in October 1996. If our estimate is causal, Fox News availability in 2000 should have **zero effect** on vote share changes in earlier elections (1988-1992, 1992-1996), when Fox News didn't exist yet.

This is a textbook **placebo test**. Finding an effect in the pre-period would suggest our controls don't fully account for selection into Fox News availability.

In [ ]:
# Placebo 1: Change in R vote share 1992-1996 (Fox News didn't exist yet)
sub1 = sample.dropna(subset=['reppresfv2p96m92'])
m_placebo1 = smf.wls(
    'reppresfv2p96m92 ~ foxnews2000 + ' + ' + '.join(available),
    data=sub1, weights=sub1['totpresvotes1996']
).fit(cov_type='cluster', cov_kwds={'groups': sub1['account2000']})

# Placebo 2: Change in R vote share 1988-1992
sub2 = sample.dropna(subset=['reppresfv2p92m88'])
m_placebo2 = smf.wls(
    'reppresfv2p92m88 ~ foxnews2000 + ' + ' + '.join(available),
    data=sub2, weights=sub2['totpresvotes1996']
).fit(cov_type='cluster', cov_kwds={'groups': sub2['account2000']})

print('Placebo Tests (Table VI): Fox News effect on EARLIER election changes')
print('=' * 65)
print(f'{"":<30} {"1996-2000":>10} {"1992-1996":>10} {"1988-1992":>10}')
print(f'{"":30} {"(real)":>10} {"(placebo)":>10} {"(placebo)":>10}')
print('-' * 65)
print(f'{"Fox News coef":<30} {m2.params["foxnews2000"]:>+10.4f} {m_placebo1.params["foxnews2000"]:>+10.4f} {m_placebo2.params["foxnews2000"]:>+10.4f}')
print(f'{"SE":<30} {m2.bse["foxnews2000"]:>10.4f} {m_placebo1.bse["foxnews2000"]:>10.4f} {m_placebo2.bse["foxnews2000"]:>10.4f}')
print(f'{"p-value":<30} {m2.pvalues["foxnews2000"]:>10.4f} {m_placebo1.pvalues["foxnews2000"]:>10.4f} {m_placebo2.pvalues["foxnews2000"]:>10.4f}')
print(f'{"N":<30} {int(m2.nobs):>10,} {int(m_placebo1.nobs):>10,} {int(m_placebo2.nobs):>10,}')
print('-' * 65)
print('\nThe real effect (1996-2000) is positive: Fox News helped Republicans.')
print('The placebo effects (earlier elections) are NOT significant:')
print('Fox News in 2000 did not "cause" vote changes before it existed.')
print('This supports the causal interpretation.')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Fox News effect across specifications (Simpson's paradox)
coefs = [m1.params['foxnews2000'], m2.params['foxnews2000']]
ses = [m1.bse['foxnews2000'], m2.bse['foxnews2000']]
labels = ['No controls', 'Census controls']

ax = axes[0]
ax.barh(range(len(coefs)), coefs, xerr=[1.96*s for s in ses],
        color=['#e74c3c', '#3498db'], capsize=5, height=0.5, alpha=0.8)
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Fox News effect on R vote share change (pp)', fontsize=11)
ax.set_title('Table IV: Fox News Effect', fontsize=13)
ax.grid(True, alpha=0.3, axis='x')

# Panel B: Real vs placebo - use t-statistics instead of raw coefficients
# This is the honest comparison: how many SEs from zero?
ax = axes[1]
test_coefs = [m2.params['foxnews2000'], m_placebo1.params['foxnews2000'], m_placebo2.params['foxnews2000']]
test_ses = [m2.bse['foxnews2000'], m_placebo1.bse['foxnews2000'], m_placebo2.bse['foxnews2000']]
test_pvals = [m2.pvalues['foxnews2000'], m_placebo1.pvalues['foxnews2000'], m_placebo2.pvalues['foxnews2000']]
t_stats = [c/s for c, s in zip(test_coefs, test_ses)]
test_labels = ['1996-2000\n(real)', '1992-1996\n(placebo)', '1988-1992\n(placebo)']
colors = ['#2ecc71', '#95a5a6', '#95a5a6']

bars = ax.bar(range(3), t_stats, color=colors, width=0.6, alpha=0.8)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8)
ax.axhline(y=1.96, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(y=-1.96, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.text(2.55, 1.96, 'p = 0.05', fontsize=9, color='red', alpha=0.7, va='bottom')

# Add p-value annotations
for i, (bar, pval) in enumerate(zip(bars, test_pvals)):
    height = bar.get_height()
    label = f'p = {pval:.3f}'
    if pval < 0.05:
        label += ' *'
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold' if pval < 0.05 else 'normal')

ax.set_xticks(range(3))
ax.set_xticklabels(test_labels, fontsize=10)
ax.set_ylabel('t-statistic (coefficient / SE)', fontsize=11)
ax.set_title('Placebo Tests: How Many SEs from Zero?', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Left: Without controls, the effect is negative (Simpson\'s paradox). Controls flip it positive.')
print('Right: The t-statistic measures signal strength (coefficient / SE).')
print('Only the real period (1996-2000) approaches significance. The placebos are noise.')